# Citation Linking — Dataset Builder

Builds and uploads the canonical `yurui983/citation_linking` HuggingFace dataset from the
raw benchmark result files, then populates the Argilla annotation service.

## Structure
1. Imports, config, helpers
2. Load all result files
3. Build `full` split  (4 440 rows — all records × all indexes)
4. Build `annotated` split (2 000 rows — 500 sampled refs × all indexes)
5. Push to HuggingFace Hub
6. create Argilla datasets from the `annotated` split

In [1]:
import json
import os
import random
import re
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
from datasets import Dataset, DatasetDict

# ---------------------------------------------------------------------------
# Configuration — override via environment variables or edit directly
# ---------------------------------------------------------------------------
HF_REPO          = "yurui983/citation_linking"
HF_TOKEN         = os.environ.get("HF_TOKEN")

ARGILLA_API_URL   = os.environ.get("ARGILLA_API_URL")  or "https://argilla.graphia-ssh.eu/"
ARGILLA_API_TOKEN = os.environ.get("ARGILLA_API_TOKEN") or "argilla.apikey"

RANDOM_SEED          = 42
SAMPLE_PER_SOURCE    = 100   # → 5 sources × 100 = 500 annotated refs
INDEXES              = ["openalex", "matilda", "wikidata", "opencitations"]

print("✓ Config loaded")
print(f"  HF repo       : {HF_REPO}")
print(f"  Argilla URL   : {ARGILLA_API_URL}")
print(f"  Random seed   : {RANDOM_SEED}")

✓ Config loaded
  HF repo       : yurui983/citation_linking
  Argilla URL   : https://argilla.graphia-ssh.eu/
  Random seed   : 42


In [2]:
# ---------------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------------

def load_records(path: Path) -> List[Dict]:
    """Load a JSON array or JSONL file into a list of dicts."""
    content = path.read_text(encoding="utf-8").strip()
    if content.startswith("["):
        return json.loads(content)
    return [json.loads(line) for line in content.splitlines() if line.strip()]


def infer_source(ref_id: str) -> str:
    """Infer the corpus source from a ref_id prefix."""
    if not ref_id:
        return "unknown"
    if ref_id.startswith("cex_"):
        return "cex"
    if ref_id.startswith("excite_"):
        return "excite"
    if ref_id.startswith("linkedbook_"):
        return "linkedbook"
    if ref_id.startswith("brill_"):
        return "brill"
    if re.match(r"^10\.\d+_", ref_id):
        return "legal_study_mpilhlt"
    return "unknown"


def extract_openalex_id(raw_id: Optional[str]) -> Optional[str]:
    if not raw_id:
        return None
    return raw_id.rstrip("/").split("/")[-1]


def build_matched_id(index: str, ids: Dict[str, Any]) -> Optional[str]:
    if index == "openalex":
        return extract_openalex_id(ids.get("openalex_id"))
    if index == "matilda":
        return ids.get("matilda_id")
    if index == "wikidata":
        return ids.get("wikidata_id")
    if index == "opencitations":
        omid = ids.get("omid")
        if omid:
            parts = omid.rstrip("/").split("/")
            if len(parts) >= 2:
                return "/".join(parts[-2:])
        return None
    return None


def build_matched_link(index: str, matched_id: Optional[str]) -> Optional[str]:
    fallbacks = {
        "openalex":      "https://openalex.org/",
        "matilda":       "https://matilda.science/?l=en",
        "wikidata":      "https://www.wikidata.org/wiki/Wikidata:Main_Page",
        "opencitations": "https://sparql.opencitations.net/",
    }
    if matched_id:
        if index == "openalex":
            return f"https://openalex.org/works?zoom={matched_id.lower()}"
        if index == "matilda":
            return f"https://matilda.science/work/{matched_id}"
        if index == "wikidata":
            return f"https://www.wikidata.org/wiki/{matched_id}"
        if index == "opencitations":
            return f"https://api.opencitations.net/meta/v1/metadata/omid:{matched_id}"
    return fallbacks.get(index)


def resolve_is_match(metadata: Optional[Dict[str, Any]]) -> bool:
    if not metadata:
        return False
    top_result = metadata.get("top_result") or {}
    is_match = top_result.get("is_match")
    if is_match is not None:
        return bool(is_match)
    title_sim = (top_result.get("match_details") or {}).get("title_similarity")
    return float(title_sim) >= 90.0 if title_sim is not None else False


def summarize_top_result(top_result: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    data = top_result or {}
    return {f: data.get(f) for f in ("title", "first_author", "year", "journal")}


def matched_result_to_markdown_json(m: Optional[Dict[str, Any]]) -> str:
    if not m:
        return ""
    return "```json\n" + json.dumps(m, indent=2) + "\n```"


def fix_opencitations_omid(records: List[Dict]) -> None:
    """Normalise OMID to last two path segments in-place."""
    for record in records:
        sr = record.get("search_results") or {}
        oc = (sr.get("opencitations") or {}).get("metadata_search") or {}
        top = oc.get("top_result") or {}
        ids = top.get("ids") or {}
        omid = ids.get("omid")
        if omid:
            parts = omid.rstrip("/").split("/")
            if len(parts) >= 2:
                ids["omid"] = "/".join(parts[-2:])


def records_to_rows(records: List[Dict], index: str) -> List[Dict[str, Any]]:
    """Convert raw benchmark records for one citation index into flat dataset rows."""
    rows = []
    for record in records:
        ref_id   = record.get("ref_id") or ""
        orig_str = record.get("original_string") or ""
        source   = infer_source(ref_id)

        sr       = record.get("search_results") or {}
        metadata = (sr.get(index) or {}).get("metadata_search")
        top      = metadata.get("top_result") if metadata else None
        ids      = (top or {}).get("ids") or {}

        matched_id = build_matched_id(index, ids)
        rows.append({
            "ref_id":               ref_id,
            "source":               source,
            "original_ref_string":  orig_str,
            "index":                index,
            "matched_id":           matched_id or "Not Found",
            "matched_doi":          ids.get("doi") or "",
            "matched_result":       summarize_top_result(top),
            "is_match_by_similarity": resolve_is_match(metadata),
            "matched_link":         build_matched_link(index, matched_id),
        })
    return rows


print("✓ Helper functions loaded")

✓ Helper functions loaded


## 2. Load All Result Files

In [3]:
HERE = Path(".")   # notebook runs from benchmarks/citation_linking/

# --- Original data (cex / excite / linkedbook) ---
original_main = load_records(HERE / "results_20251020_124218_limitNone_openalex_wikidata_matilda.json")
original_oc   = load_records(HERE / "results_20251022_161614_limitNone_opencitations.json")

# --- Brill KG ---
brill_main = load_records(HERE / "brill_search_result_20251106_165209_limitNone_openalex_wikidata_matilda.jsonl")
brill_oc   = load_records(HERE / "brill_search_result_20251107_074130_limitNone_opencitations.jsonl")

# --- Legal Study (MPILHLT) — auto-pick latest file ---
legal_main_files = sorted(HERE.glob("legal_study_results_*_openalex_*matilda*.json"))
legal_oc_files   = sorted(HERE.glob("legal_study_results_*_opencitations.json"))

if not legal_main_files:
    raise FileNotFoundError("No legal study openalex/wikidata/matilda results found.")
if not legal_oc_files:
    raise FileNotFoundError("No legal study opencitations results found.")

legal_main = load_records(legal_main_files[-1])
legal_oc   = load_records(legal_oc_files[-1])

# Fix OpenCitations OMID format in all OC files
for recs in (original_oc, brill_oc, legal_oc):
    fix_opencitations_omid(recs)

# --- Group by citation index ---
# openalex / matilda / wikidata all live in the same "main" files;
# opencitations has its own separate files.
index_records: Dict[str, List[Dict]] = {
    "openalex":      original_main + brill_main + legal_main,
    "matilda":       original_main + brill_main + legal_main,
    "wikidata":      original_main + brill_main + legal_main,
    "opencitations": original_oc   + brill_oc   + legal_oc,
}

print("Record counts per index:")
for idx, recs in index_records.items():
    from collections import Counter
    src_counts = Counter(infer_source(r.get("ref_id", "")) for r in recs)
    print(f"  {idx:15s}: {len(recs):4d}  {dict(src_counts)}")

Record counts per index:
  openalex       : 1110  {'cex': 459, 'excite': 193, 'linkedbook': 107, 'brill': 200, 'legal_study_mpilhlt': 151}
  matilda        : 1110  {'cex': 459, 'excite': 193, 'linkedbook': 107, 'brill': 200, 'legal_study_mpilhlt': 151}
  wikidata       : 1110  {'cex': 459, 'excite': 193, 'linkedbook': 107, 'brill': 200, 'legal_study_mpilhlt': 151}
  opencitations  : 1110  {'cex': 459, 'excite': 193, 'linkedbook': 107, 'brill': 200, 'legal_study_mpilhlt': 151}


## 3. Build `full` Split

All 1 110 references × 4 citation indexes = **4 440 rows**.
Each row carries a `source` (cex / excite / linkedbook / brill / legal_study_mpilhlt)
and an `index` (openalex / matilda / wikidata / opencitations) column.

In [4]:
all_rows: List[Dict[str, Any]] = []

for idx_name, records in index_records.items():
    all_rows.extend(records_to_rows(records, idx_name))

ds_full = Dataset.from_list(all_rows)

print(f"full split: {len(ds_full)} rows")
print(ds_full)

# Sanity check: row counts per index and per source
df_full = ds_full.to_pandas()
print("\nRows per index:")
print(df_full["index"].value_counts().to_string())
print("\nRows per source:")
print(df_full["source"].value_counts().to_string())
print("\nRows per source × index:")
display(df_full.groupby(["source", "index"]).size().unstack(fill_value=0))

full split: 4440 rows
Dataset({
    features: ['ref_id', 'source', 'original_ref_string', 'index', 'matched_id', 'matched_doi', 'matched_result', 'is_match_by_similarity', 'matched_link'],
    num_rows: 4440
})

Rows per index:
index
openalex         1110
matilda          1110
wikidata         1110
opencitations    1110

Rows per source:
source
cex                    1836
brill                   800
excite                  772
legal_study_mpilhlt     604
linkedbook              428

Rows per source × index:


index,matilda,openalex,opencitations,wikidata
source,,,,
brill,200,200,200,200
cex,459,459,459,459
excite,193,193,193,193
legal_study_mpilhlt,151,151,151,151
linkedbook,107,107,107,107


## 4. Build `annotated` Split

Sample **100 ref_ids per source** (fixed seed → reproducible) → 500 unique refs.
Those same 500 ref_ids are then taken from every citation index → **2 000 rows** total.

This ensures annotators can compare the same reference across all four indexes.

In [5]:
rng = random.Random(RANDOM_SEED)

# Collect unique ref_ids per source (use only one index as reference — all share the same refs)
ref_ids_by_source: Dict[str, List[str]] = defaultdict(list)
for record in index_records["openalex"]:   # any index works; ref_ids are shared
    ref_id = record.get("ref_id") or ""
    ref_ids_by_source[infer_source(ref_id)].append(ref_id)

print("Available ref_ids per source:")
for src, ids in sorted(ref_ids_by_source.items()):
    print(f"  {src:25s}: {len(ids):4d}")

# Sample SAMPLE_PER_SOURCE from each source
sampled_ref_ids: set[str] = set()
for src, ids in ref_ids_by_source.items():
    n = min(SAMPLE_PER_SOURCE, len(ids))
    sampled = rng.sample(ids, n)
    sampled_ref_ids.update(sampled)
    print(f"  sampled {n} from '{src}'")

print(f"\nTotal sampled ref_ids: {len(sampled_ref_ids)}")

# Build annotated rows: same sampled ref_ids across ALL indexes
annotated_rows: List[Dict[str, Any]] = []
for idx_name, records in index_records.items():
    for record in records:
        if record.get("ref_id") in sampled_ref_ids:
            annotated_rows.extend(records_to_rows([record], idx_name))

ds_annotated = Dataset.from_list(annotated_rows)

print(f"\nannotated split: {len(ds_annotated)} rows")
df_ann = ds_annotated.to_pandas()
print("\nRows per index:")
print(df_ann["index"].value_counts().to_string())
print("\nRows per source:")
print(df_ann["source"].value_counts().to_string())

Available ref_ids per source:
  brill                    :  200
  cex                      :  459
  excite                   :  193
  legal_study_mpilhlt      :  151
  linkedbook               :  107
  sampled 100 from 'cex'
  sampled 100 from 'excite'
  sampled 100 from 'linkedbook'
  sampled 100 from 'brill'
  sampled 100 from 'legal_study_mpilhlt'

Total sampled ref_ids: 500

annotated split: 2000 rows

Rows per index:
index
openalex         500
matilda          500
wikidata         500
opencitations    500

Rows per source:
source
cex                    400
excite                 400
linkedbook             400
brill                  400
legal_study_mpilhlt    400


## 5. Push to HuggingFace Hub

Replaces the existing dataset with the new two-split structure.

In [6]:
from huggingface_hub import login

login(token=HF_TOKEN)

dataset_dict = DatasetDict({
    "full":       ds_full,
    "annotated":  ds_annotated,
})

print(dataset_dict)

dataset_dict.push_to_hub(HF_REPO, private=False)
print(f"\n✓ Dataset pushed to https://huggingface.co/datasets/{HF_REPO}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


DatasetDict({
    full: Dataset({
        features: ['ref_id', 'source', 'original_ref_string', 'index', 'matched_id', 'matched_doi', 'matched_result', 'is_match_by_similarity', 'matched_link'],
        num_rows: 4440
    })
    annotated: Dataset({
        features: ['ref_id', 'source', 'original_ref_string', 'index', 'matched_id', 'matched_doi', 'matched_result', 'is_match_by_similarity', 'matched_link'],
        num_rows: 2000
    })
})


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        : 100%|##########|  300kB /  300kB            


✓ Dataset pushed to https://huggingface.co/datasets/yurui983/citation_linking


## 6. Argilla — Delete Existing & Re-create from `annotated` Split

Deletes the four existing `citation_linking_*` datasets and re-creates them
with the 500 annotated records per citation index.

In [7]:
import argilla as rg

client = rg.Argilla(api_key=ARGILLA_API_TOKEN, api_url=ARGILLA_API_URL)

# --- Delete existing datasets ---
print("Deleting existing Argilla datasets...")
for idx_name in INDEXES:
    dataset_name = f"citation_linking_{idx_name}"
    try:
        existing = client.datasets(name=dataset_name)
        if existing:
            existing.delete()
            print(f"  ✓ Deleted {dataset_name}")
        else:
            print(f"  – {dataset_name} not found, skipping")
    except Exception as e:
        print(f"  – {dataset_name}: {e}")

print("\nDone deleting.")

Deleting existing Argilla datasets...
  ✓ Deleted citation_linking_openalex
  ✓ Deleted citation_linking_matilda
  ✓ Deleted citation_linking_wikidata
  – citation_linking_opencitations not found, skipping

Done deleting.


/opt/homebrew/Caskroom/miniforge/base/envs/citation_index/lib/python3.11/site-packages/argilla/client.py:356: UserWarning: Dataset with name 'citation_linking_opencitations' not found in workspace 'default'
  warnings.warn(f"Dataset with name {name!r} not found in workspace {workspace.name!r}")


In [8]:
ANNOTATION_GUIDELINES = """
# Annotation Guidelines: Citation Linking

![Annotation Guidelines](https://raw.githubusercontent.com/odoma-ch/ssh-citation-index/refs/heads/main/benchmarks/citation_linking/argilla-decision-flow-TD.png)

## What the fields mean
- **ref_id**: Internal identifier of the reference.
- **original_ref_string**: The raw reference text extracted from the document.
- **matched_id**: Provider-specific ID of the candidate match (or "Not Found").
  - OpenAlex: W1234567890
  - Wikidata: Q12345
  - Matilda: last path segment in /work/<id>
  - OpenCitations: OMID last path segment (e.g., br/06210459208)
- **matched_doi**: DOI of the candidate match (if available).
- **matched_result**: Compact summary of the candidate (title, first_author, year, journal).
- **is_match_by_similarity**: Heuristic guess — for context only.
- **matched_link**: Link to view the candidate on the index site.

## What you need to do
1. Check if the candidate is the **same work** as the reference.
   - Compare title, first author surname, year, and DOI (if present).
   - Use matched_link to verify on the index page.
2. If the candidate is **incorrect or missing**, find the correct record and paste its ID in `correct_id`.
3. If **no record exists** in this index, select "No match".

## How to answer
| Situation | is_match_correct | No match | correct_id |
|-----------|-----------------|----------|------------|
| Candidate is correct | true | false | (blank) |
| Candidate is wrong but record exists | false | false | correct ID |
| No record in this index | false | true | (blank) |

Notes: Minor formatting differences are fine. Provide only the ID (not a URL).
"""

argilla_settings = rg.Settings(
    fields=[
        rg.TextField(name="ref_id",               title="Reference ID"),
        rg.TextField(name="original_ref_string",  title="Reference String"),
        rg.TextField(name="source",               title="Corpus Source"),
        rg.TextField(name="matched_id",           title="Matched ID"),
        rg.TextField(name="matched_doi",          title="Matched DOI"),
        rg.TextField(name="matched_result",       title="Matched Result", use_markdown=True),
        rg.TextField(name="is_match_by_similarity", title="Is Match by Similarity"),
        rg.TextField(name="matched_link",         title="Matched Link",   use_markdown=True),
    ],
    guidelines=ANNOTATION_GUIDELINES,
    questions=[
        rg.LabelQuestion(
            name="is_match_correct",
            title="Is the matched result correct?",
            labels=["true", "false"],
            description="'true' if the candidate is the same publication as the reference.",
            required=True,
        ),
        rg.TextQuestion(
            name="correct_id",
            title="If incorrect, provide the correct ID",
            description="Provider-specific ID of the correct match. Leave blank if correct.",
            required=False,
        ),
        rg.LabelQuestion(
            name="no_match",
            title="Does this reference have no match in this index?",
            labels=["true", "false"],
            required=True,
        ),
    ],
)

print("✓ Argilla settings defined")

✓ Argilla settings defined


In [9]:
# Re-create one Argilla dataset per citation index from the annotated split
df_annotated = ds_annotated.to_pandas()

print("Creating Argilla datasets from annotated split...")

for idx_name in INDEXES:
    dataset_name = f"citation_linking_{idx_name}"
    print(f"\nCreating {dataset_name}...")

    dataset = rg.Dataset(name=dataset_name, settings=argilla_settings)
    dataset.create()

    subset = df_annotated[df_annotated["index"] == idx_name]
    print(f"  Records to upload: {len(subset)}")

    records_to_log = []
    for _, row in subset.iterrows():
        matched_raw = row.get("matched_result") or {}
        matched_text = (
            matched_result_to_markdown_json(matched_raw)
            if isinstance(matched_raw, dict)
            else str(matched_raw or "")
        )

        is_match_val = row.get("is_match_by_similarity")
        is_match_str = (
            ("true" if is_match_val else "false")
            if isinstance(is_match_val, bool)
            else str(is_match_val if is_match_val is not None else "")
        )

        matched_link_val = row.get("matched_link") or ""
        matched_link_md  = f"[{matched_link_val}]({matched_link_val})" if matched_link_val else ""

        records_to_log.append({
            "ref_id":                row.get("ref_id")               or "",
            "original_ref_string":   row.get("original_ref_string")  or "",
            "source":                row.get("source")               or "",
            "matched_id":            row.get("matched_id")           or "Not Found",
            "matched_doi":           row.get("matched_doi")          or "",
            "matched_result":        matched_text,
            "is_match_by_similarity": is_match_str,
            "matched_link":          matched_link_md,
            "metadata": {
                "ref_id":  row.get("ref_id"),
                "source":  row.get("source"),
                "index":   idx_name,
            },
        })

    dataset.records.log(records=records_to_log)
    print(f"  ✓ Uploaded {len(records_to_log)} records to {dataset_name}")

print("\n✓ All Argilla datasets created successfully!")

Creating Argilla datasets from annotated split...

Creating citation_linking_openalex...


/opt/homebrew/Caskroom/miniforge/base/envs/citation_index/lib/python3.11/site-packages/argilla/datasets/_resource.py:264: UserWarning: Workspace not provided. Using default workspace: default id: d4a1881b-5e03-4d0b-b42e-d18e4f94b1af
  warnings.warn(f"Workspace not provided. Using default workspace: {workspace.name} id: {workspace.id}")
/opt/homebrew/Caskroom/miniforge/base/envs/citation_index/lib/python3.11/site-packages/argilla/records/_mapping/_mapper.py:89: UserWarning: Keys ['metadata'] in data are not present in the mapping and will be ignored.
  warnings.warn(f"Keys {unknown_keys} in data are not present in the mapping and will be ignored.")


  Records to upload: 500


Sending records...: 2batch [00:06,  3.19s/batch]                    


  ✓ Uploaded 500 records to citation_linking_openalex

Creating citation_linking_matilda...
  Records to upload: 500


...

  ✓ Uploaded 500 records to citation_linking_matilda

Creating citation_linking_wikidata...
  Records to upload: 500


Sending records...: 2batch [00:05,  2.87s/batch]                    


  ✓ Uploaded 500 records to citation_linking_wikidata

Creating citation_linking_opencitations...
  Records to upload: 500


Sending records...: 2batch [00:05,  2.79s/batch]                    

  ✓ Uploaded 500 records to citation_linking_opencitations

✓ All Argilla datasets created successfully!
